# 4. Analyzer custom e input per un LLM

Usa lo schema in [modelli/nota_spese.json](../modelli/nota_spese.json) per la nota spese di Giulia Bianchi. Risultati attesi: progetto LAB-CU-042, totale EUR 580, anticipo EUR 200, rimborso EUR 380, stato Approvata. L'ID custom arriva da `.env`.

Documento: [esercizio4.pdf](../file/esercizio4.pdf).

Soluzione completa: esegui le celle in ordine con il kernel del venv. Ogni notebook parte da zero: nessun `main()`, `%run` o stato condiviso con gli altri esercizi.

[Traccia da svolgere](../esercizi/04-output-llm.ipynb).

## 1. Import e configurazione

Esegui dalla cartella `soluzioni`. `load_dotenv(r"..\.env", override=True)` carica il file noto e applica i suoi valori. I risultati vengono soltanto stampati.

In [1]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient, to_llm_input
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from azure.ai.contentunderstanding.models import ContentAnalyzer

In [ ]:
load_dotenv(r"..\.env", override=True)
DOCUMENTO = Path(r"..\file\esercizio4.pdf")
document_bytes = DOCUMENTO.read_bytes()

# Legge la definizione JSON, non il risultato di un'analisi.
definizione = json.loads(
    Path(r"..\modelli\nota_spese.json").read_text(encoding="utf-8")
)

## 2. Client ed esecuzione

`ContentAnalyzer(definizione)` usa direttamente il JSON, senza verifiche sui deployment. `begin_create_analyzer` riceve ID, definizione e `allow_replace=True`; `begin_analyze_binary` riceve ID e byte PDF. `result()` attende il completamento.

`with` chiude client e credenziale; `finally` elimina l'analyzer dopo l'analisi, anche se fallisce. Usa solo ID dedicati al laboratorio: `allow_replace=True` sostituisce l'analyzer esistente. Non eseguire due copie dello stesso notebook contemporaneamente. Un riavvio forzato del kernel puo impedire la pulizia.

In [ ]:
# Crea client e credenziale; with li chiude anche in caso di errore.
with DefaultAzureCredential() as credential, ContentUnderstandingClient(
    endpoint=os.environ["AZURE_CONTENTUNDERSTANDING_ENDPOINT"], credential=credential,
) as client:
    # Crea l'analyzer dal JSON; allow_replace aggiorna l'ID del laboratorio.
    client.begin_create_analyzer(
        analyzer_id="spese_custom", resource=ContentAnalyzer(definizione), allow_replace=True,
    ).result()
    try:
        # Analizza i byte del PDF con l'analyzer indicato e attende il risultato.
        result = client.begin_analyze_binary(
            analyzer_id="spese_custom", binary_input=document_bytes,
        ).result()
    finally:
        # Elimina l'analyzer temporaneo identificato da analyzer_id.
        client.delete_analyzer(analyzer_id="spese_custom")

## 3. Osserva il risultato

Mostra il risultato della chiamata Azure. `as_dict()` converte gli oggetti SDK in dizionari; `json.dumps(..., ensure_ascii=False, indent=2)` produce JSON leggibile. `print` mostra il risultato senza salvarlo su disco.

In [4]:
for content in result.contents:
    fields = content.as_dict()["fields"]
    print(json.dumps(fields, ensure_ascii=False, indent=2))

{
  "Dipendente": {
    "type": "string",
    "valueString": "Giulia Bianchi",
    "spans": [
      {
        "offset": 71,
        "length": 14
      }
    ],
    "confidence": 0.965,
    "source": "D(1,1.7051,1.8232,2.5683,1.8190,2.5691,1.9781,1.7059,1.9823)"
  },
  "Progetto": {
    "type": "string",
    "valueString": "LAB-CU-042",
    "spans": [
      {
        "offset": 103,
        "length": 10
      }
    ],
    "confidence": 0.967,
    "source": "D(1,2.0151,2.0465,2.8104,2.0405,2.8116,2.1981,2.0163,2.2040)"
  },
  "TotaleSpese": {
    "type": "number",
    "valueNumber": 580,
    "spans": [
      {
        "offset": 533,
        "length": 6
      }
    ],
    "confidence": 0.963,
    "source": "D(1,5.5604,5.9276,6.0002,5.9217,6.0002,6.0595,5.5604,6.0692)"
  },
  "Anticipo": {
    "type": "number",
    "valueNumber": 200,
    "spans": [
      {
        "offset": 591,
        "length": 6
      }
    ],
    "confidence": 0.964,
    "source": "D(1,5.5604,6.3479,6.0002,6.3479,6.000

## 4. Prepara il contesto per un LLM

`to_llm_input(result)` combina campi e Markdown localmente, senza un'altra chiamata Azure. `include_markdown=False` conserva solo i campi; `include_fields=False` conserva solo il testo.

In [5]:
# Converte campi e Markdown del risultato in un unico testo per un LLM.
contesto = to_llm_input(result)
print(contesto)

print("SOLO CAMPI")
print(to_llm_input(result, include_markdown=False))
print("\nSOLO MARKDOWN")
print(to_llm_input(result, include_fields=False))

---
mimeType: application/pdf
metadata:
  contentType: application/pdf
  pageCount: '2'
  author: Content Understanding Lab
  title: Esercizio 4 - documento sintetico
  createdAt: '2026-10-05T16:43:52+02:00'
pages: 1-2
fields:
  Dipendente: Giulia Bianchi
  Progetto: LAB-CU-042
  TotaleSpese: 580
  Anticipo: 200
  RimborsoDovuto: 380
  StatoApprovazione: Approvata
  Sintesi: Trasferta a Torino per un workshop di due giornate sulla digitalizzazione dei documenti del progetto LAB-CU-042. Rimborso dovuto pari a EUR 380.00 al netto dell'anticipo.
---
<!-- InputPageNumber: 1 -->

# NORD ESEMPIO S.R.L.

Nota spese trasferta | NS-2026-004

Dipendente: Giulia Bianchi
Codice progetto: LAB-CU-042

Destinazione: Torino

Periodo: 2026-09-14 / 2026-09-16

Finalita della trasferta

Workshop di due giornate sulla digitalizzazione dei documenti del progetto LAB-CU-042.


## Riepilogo economico - EUR


<table>
<tr>
<td>Voce</td>
<td>Importo EUR</td>
</tr>
<tr>
<td>Pernottamenti</td>
<td>440.00</td>
</t